In [1]:

from datasets import load_dataset, load_from_disk
import torch
import os


In [ ]:
# !pip install --upgrade transformers

In [2]:
train = load_from_disk('/content/train')
validation = load_from_disk('/content/validation')

In [3]:
train, validation

(Dataset({
     features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 14732
 }),
 Dataset({
     features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
     num_rows: 818
 }))

In [4]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import os

class ModelTrainer:
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained('google/pegasus-cnn_dailymail')
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained('google/pegasus-cnn_dailymail').to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        #loading data
        train = load_from_disk('/content/train')
        validation = load_from_disk('/content/validation')


        trainer_args = TrainingArguments(
                          output_dir='/content/model',
                          num_train_epochs=1,
                          per_device_train_batch_size=2,
                          per_device_eval_batch_size=4,
                          gradient_accumulation_steps=16,
                          warmup_steps=500,
                          weight_decay=0.01,
                          logging_steps=10,
                          eval_strategy="steps",
                          eval_steps=500,
                          save_steps=1e6
)



        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=train,
                  eval_dataset=validation)

        trainer.train()
        trainer.save_model('/content/pegasus_model')

In [5]:
trainer = ModelTrainer()
trainer.train()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: skff2221 (skff2221-national-institute-of-technology-tiruchirappalli) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:4034: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


In [12]:
#pipeline
from transformers import pipeline
pipe=pipeline('summarization',model='/content/pegasus_model')
gen_kwargs={'length_penalty':0.8,'num_beams':8,'max_length':128}

text="""Ravi: Hey Meera, have you finished the project report?
Meera: Almost, just polishing the conclusion. Why?
Ravi: The professor asked me this morning, and I wasn’t sure.
Meera: Don’t worry, I’ll send it by evening.
Ravi: Great. By the way, are you joining the workshop tomorrow?
Meera: Yes, it’s on data science, right?
Ravi: Exactly. I think it’ll help with our internship applications.
Meera: True, learning practical skills is always useful.
Ravi: Perfect, let’s attend together.
Meera: Done! And I’ll mail the report soon.
"""

print(pipe(text, **gen_kwargs)[0]['summary_text'])

Device set to use cuda:0


Meera has finished the project report. She will send it to Ravi by evening. Meera will attend a workshop on data science tomorrow. She will mail the report soon.


In [11]:
from huggingface_hub import HfApi, create_repo, notebook_login
import os

# Log in to Hugging Face Hub
notebook_login()

api = HfApi()

# Define the repository ID
repo_id = "sanjeevnits24/summerizer"

# Create the repository if it doesn't exist
try:
    create_repo(repo_id, repo_type="model", exist_ok=True)
except Exception as e:
    print(f"Could not create repository: {e}")


# Upload the folder
try:
    api.upload_folder(
        folder_path="/content/pegasus_model",
        repo_id=repo_id,
        repo_type="model"
    )
    print(f"Successfully uploaded to {repo_id}")
except Exception as e:
    print(f"Could not upload folder: {e}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /content/pegasus_model/spiece.model   : 100%|##########| 1.91MB / 1.91MB            

  ...ent/pegasus_model/model.safetensors:   0%|          | 4.65MB / 2.28GB            

  ...ent/pegasus_model/training_args.bin:   6%|6         |   358B / 5.78kB            

Successfully uploaded to sanjeevnits24/summerizer


In [18]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Replace with your model’s repo
model_name = "sanjeevnits24/summerizer"

# Load tokenizer (if available)
tokenizer_s = AutoTokenizer.from_pretrained(model_name)

# Load model
model_s = AutoModelForSeq2SeqLM.from_pretrained(model_name)


generation_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

In [19]:
from transformers import pipeline
pipe=pipeline('summarization',model=model_s, tokenizer=tokenizer_s)
gen_kwargs={'length_penalty':0.8,'num_beams':8,'max_length':128}

text="""Ravi: Hey Meera, have you finished the project report?
Meera: Almost, just polishing the conclusion. Why?
Ravi: The professor asked me this morning, and I wasn’t sure.
Meera: Don’t worry, I’ll send it by evening.
Ravi: Great. By the way, are you joining the workshop tomorrow?
Meera: Yes, it’s on data science, right?
Ravi: Exactly. I think it’ll help with our internship applications.
Meera: True, learning practical skills is always useful.
Rera: Perfect, let’s attend together.
Meera: Done! And I’ll mail the report soon.
"""

print(pipe(text, **gen_kwargs)[0]['summary_text'])

Device set to use cuda:0


Meera has finished the project report. She will send it to Ravi by evening. Meera will attend a workshop on data science tomorrow with Rera.


In [ ]:
!zip pegasus_model.zip -r /content/pegasus_model/

  adding: content/pegasus_model/ (stored 0%)
  adding: content/pegasus_model/training_args.bin (deflated 54%)
  adding: content/pegasus_model/tokenizer.json (deflated 78%)
  adding: content/pegasus_model/config.json (deflated 61%)
  adding: content/pegasus_model/spiece.model (deflated 50%)
  adding: content/pegasus_model/model.safetensors (deflated 7%)
  adding: content/pegasus_model/special_tokens_map.json (deflated 82%)
  adding: content/pegasus_model/tokenizer_config.json (deflated 94%)
  adding: content/pegasus_model/generation_config.json (deflated 44%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp /content/your_file.zip /content/drive/MyDrive/


Mounted at /content/drive
cp: cannot stat '/content/your_file.zip': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# trainer.save_model('/content/model')

In [ ]:
# from huggingface_hub import HfApi, create_repo, notebook_login
# import os

# # Log in to Hugging Face Hub
# notebook_login()

# api = HfApi()

# # Define the repository ID
# repo_id = "sanjeevnits24/text-summerization"

# # Create the repository if it doesn't exist
# try:
#     create_repo(repo_id, repo_type="model", exist_ok=True)
# except Exception as e:
#     print(f"Could not create repository: {e}")


# # Upload the folder
# try:
#     api.upload_folder(
#         folder_path="/content/model",
#         repo_id=repo_id,
#         repo_type="model"
#     )
#     print(f"Successfully uploaded to {repo_id}")
# except Exception as e:
#     print(f"Could not upload folder: {e}")

In [ ]:
# !wget https://huggingface.co/sanjeevnits24/text-summerization/resolve/main/model.safetensors -P /content/